In [33]:
using LinearAlgebra

# Projeto 2 - Jacobi e Gauss-Seidel

Modelo de entrega.

Respeite as divisões em Markdown para facilitar a navegação no relatório.

## Questão 1a: Implementação

Implemente os métodos de Jacobi e Gauss-Seidel para resolver o sistema linear $Ax = b$.

In [139]:
#Função que retorna o módulo da diferença entre as normas dos nossos vetores, para conseguimos dimensionar o erro da nossa solução
get_erro(x_real, x_aprox) = abs(norm(x_real) - norm(x_aprox))

get_erro (generic function with 1 method)

In [140]:
@time get_erro([1,1,2], [1,1,1])

0.7174389352143007

  0.000004 seconds (4 allocations: 160 bytes)


### JacobiMethod(A, b, n)

Retorna uma aproximação da solução do sistema linear $Ax = b$, utilizando o método iterativo de Jacobi.

O método de Jacobi decompõe a matriz em $A = D + R$ e utiliza a iteração: $x^{(k+1)} = D^{-1}(b - Rx^{(k)})$

#### Argumentos
- `A::Matrix`: Matriz de coeficientes.
- `b::Vector`: Vetor resultante.
- `n::Int`: Número máximo de iterações.

#### Retorno
- `x::Vector`: Aproximação da solução x após as n iterações.


In [197]:
function JacobiMethod(A::Matrix , b:: Vector, ϵ:: Float64, n:: Int = 0)
    x_real = A\b

    m = size(A,1)
    x = ones(m) #Chute inicial

    if n == 0
        erro = 1

        while erro >= ϵ
            xᵏ = copy(x)

            for i in 1:m
                diag_inv = 1/A[i,i]
                d = diag_inv * b[i] #Cálculo do D⁻¹[i,i]*b[i]

                soma = 0
                for j in 1:m #Cálculo do D⁻¹Rx
                    if j != i
                        soma = soma + diag_inv * A[i,j] * xᵏ[j] #∑D⁻¹[i,i] * A[i,j] * xᵏ[j]
                    end
                end

                x[i] = d - soma # x =  D⁻¹b - D⁻¹Rx 
            end

            erro = get_erro(x, xᵏ)
        end

    else
        for k in 1:n
            xᵏ = copy(x)

            for i in 1:m
                diag_inv = 1/A[i,i]
                d = diag_inv * b[i] #Cálculo do D⁻¹[i,i]*b[i]

                soma = 0
                for j in 1:m #Cálculo do D⁻¹Rx
                    if j != i
                        soma = soma + diag_inv * A[i,j] * xᵏ[j] #∑D⁻¹[i,i] * A[i,j] * xᵏ[j]
                    end
                end

                x[i] = d - soma # x =  D⁻¹b - D⁻¹Rx 

            end
        end
    end
        
    return x
end

JacobiMethod (generic function with 4 methods)

In [198]:
function GaussSeidelMethod(A::Matrix , b:: Vector, ϵ:: Float64, n:: Int = 0)
    m = size(A,1)
    x = ones(m) #Chute inicial

    if n == 0
        erro = 1

        while erro >= ϵ
            xᵏ = copy(x)

            for i in 1:m
                diag_inv = 1/A[i,i]
                d = diag_inv * b[i] #Cálculo do D⁻¹[i,i]*b[i]

                
                soma = 0
                for j in 1:m #Cálculo do D⁻¹Rx
                    if j != i
                        soma = soma + diag_inv * A[i,j] * x[j]
                    end
                end

                x[i] = d - soma
                
            end

            erro = get_erro(x, xᵏ)
        end

    else
        for k in 1:n

            for i in 1:m
                diag_inv = 1/A[i,i]
                d = diag_inv * b[i] #Cálculo do D⁻¹[i,i]*b[i]

                
                soma = 0
                for j in 1:m #Cálculo do D⁻¹Rx
                    if j != i
                        soma = soma + diag_inv * A[i,j] * x[j]
                    end
                end

                x[i] = d - soma
                
            end
        end
    end
    
    return x
end

GaussSeidelMethod (generic function with 4 methods)

Qual deveria ser a complexidade computacional do código que você escreveu?

Para melhor compreensão da complexidade das nossas funções, escolhi não utilizar algumas facilidades do Julia. Dessa forma, podemos perceber que em ambos possuímos 3 for's aninhados, o primeiro que vai até n e os dois últimos até m, o que configura uma complexidade $O(n*m^2)$.

## Questão 1b: Testes
Teste com matrizes $2 \times 2$ e $3 \times 3$, e compare graficamente a velocidade de convergência dos dois métodos.

In [199]:
function getComparativo(size:: Int, n:: Float64)
    A = rand(Int, (size,size))
    b = rand(Int, size)

    x_real = A\b

    x_j = JacobiMethod(A, b, n)
    x_gs = GaussSeidelMethod(A, b, n)

    ϵ_j = get_erro(x_real, x_j)
    ϵ_gs = get_erro(x_real, x_gs)

    println("Comparativo")

    println("x_real:", x_real)

    println("x_jacobiana:", x_j)
    println("erro:", ϵ_j)

    println("x_gauss_seidel:", x_gs)
    println("erro:", ϵ_gs)

end

getComparativo (generic function with 2 methods)

In [200]:
A = [10 3; 3 10]
b = [4, 5]

x_real = A\b

2-element Vector{Float64}:
 0.27472527472527475
 0.4175824175824176

In [239]:
getComparativo(2, 0.3)

Comparativo
x_real:[1.142488974012454, 0.5335103363755515]
x_jacobiana:[3.95889790499864, -8.687032263150773]
erro:8.2856730755392
x_gauss_seidel:[-90.63485425815001, -82.57410343105926]
erro:121.3488668909312


In [248]:
JacobiMethod(A,b,16)

2-element Vector{Float64}:
 0.2747252778473446
 0.4175824200895343

In [247]:
GaussSeidelMethod(A,b,5)

2-element Vector{Float64}:
 0.27471381100000003
 0.4175858567

In [ ]:
# Código gerando gráficos

(Comentários)

## Questão 1c: Matrizes maiores
Teste com matrizes maiores.  Estes métodos funcionam para matrizes do tipo `rand(m,m)`?

In [ ]:
# Código

(Comentários)

## Questão 1d: Matrizes maiores ainda!
Quão grande deve ser $k(m)$ para funcionar para 95\% das matrizes do tipo `randn(m,m)` + $ k(m) \cdot I_m$?  Quais velocidades de convergência você observa para este $k(m)$?

In [ ]:
# Código (gerando possivelmente gráficos ou tabelas)

(Comentários)

## Questão 1e: Jacobi vs Gauss-Seidel
Encontre uma matriz $A$ $4 \times 4$ para a qual o método de Jacobi tenha melhor convergência que o método de Gauss-Seidel.

(Explique suas ideias, verifique abaixo)

In [ ]:
# Código

## Questão 1f: Perturbações
O que acontece se você somar uma matriz aleatória e pequena à matriz $A$ acima?  O que acontece se esta perturbação acontecer apenas fora da diagonal?

In [ ]:
# Código

(Comentários)